# Описание подхода

для решения использовал два внешних OCR датасета: MJSynth и bilingual ru en. Для каждого изображения создавал два варианта: исходный с label = 0 и повернутый на 180 градусов с label = 1.
В качестве модели использовал pretrained MobileNetV3-Small. Сначала обучил модели отдельно на каждом датасете и проверил их на данных друг друга. После этого обучил итоговую модель на смеси двух датасетов
Чтобы сделать синтетические данные более похожими на реальные OCR кропы, добавил аугментации: блюр, изменение яркости и контраста, небольшие геометрические и перспективные искажения
При инференсе использовал тта: делал предсказание для исходного изображения и для версии, повернутой на 180 градусов, после чего объединял вероятности
Основная метрика валидации 1 - Brier Score
    
  

# импорты 

In [1]:
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
from pathlib import Path
import pandas as pd

SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

cuda


# Подготовка данных


## общая обработка данных


### создаем примеры 0 и 180

так как во внешних датасетах нет нужной разметки ориентации, для каждого изображения создаем две экземпляра: исходную с label = 0 и повернутую на 180 градусов с label = 1. Эти пары дальше передаем в Dataset и DataLoader, чтобы модель обучалась различать две ориентации текста

In [2]:
def create_orientation_samples(files):
    samples = []

    for path in files:
        samples.append((path, 0))
        samples.append((path, 1))

    return samples



### resize и padding


приводим изображения к единому размеру 48 на 384, сохраняя исходное соотношение сторон, чтобы не искажать текст. свободное пространство заполняем padding. нужно, чтобы дальше DataLoader мог собирать изображения одинакового размера в батчи и передавать их в модель.



In [3]:
from PIL import Image

def resize_with_padding(img, target_h=48, target_w=384):
    w, h = img.size
    scale = min(target_w / w, target_h / h)
    new_w = int(w * scale)
    new_h = int(h * scale)
    img = img.resize((new_w, new_h), Image.Resampling.BILINEAR)
    canvas = Image.new("RGB", (target_w, target_h), (128, 128, 128))

    left = (target_w - new_w) // 2
    top = (target_h - new_h) // 2
    canvas.paste(img, (left, top))

    return canvas

### dataset


создаем датасет. Загружаем изображение, при label = 1 поворачиваем его на 180 градусов, применяем resize с padding, переводим в tensor и нормализуем. Нормализацию используем под pretrained веса MobileNetV3-Small. После этого DataLoader будет собирать подготовленные изображения в батчи для обучения и валидации 


для обучения используем аугментации, чтобы приблизить синтетические данные к реальным OCR-кропам. Добавляем размытие, изменение яркости и контраста, небольшие перспективные и геометрические искажения. Это помогает модели лучше работать на изображениях, которые отличаются от обучающих данных

In [4]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image

train_transform = transforms.Compose([
    transforms.RandomApply([
        transforms.GaussianBlur(
            kernel_size=5,
            sigma=(0.4, 2.5)
        )
    ], p=0.6),

    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.5,
            contrast=0.5,
            saturation=0.3
        )
    ], p=0.7),

    transforms.RandomPerspective(
        distortion_scale=0.2,
        p=0.3,
        fill=(128, 128, 128)
    ),

    transforms.RandomAffine(
        degrees=6,
        translate=(0.04, 0.06),
        fill=(128, 128, 128)
    ),

    transforms.RandomGrayscale(p=0.2),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


val_transform = transforms.Compose([
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class OrientationDataset(Dataset):

    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):

        path, label = self.samples[i]

        img = Image.open(path).convert("RGB")

        if label == 1:
            img = img.rotate(180)

        img = resize_with_padding(img)

        img = self.transform(img)

        return img, label

## mjsynth

### Загрузка mjsynth

загружаем внешний датасет MJSynth, так как в задании нет обучающих данных. Используем его как источник текстовых изображений для обучения модели



In [5]:
from datasets import load_dataset

mjsynth = load_dataset(
    "priyank-m/MJSynth_text_recognition",
    split="train",
    streaming=True
)

c:\Users\Aryan\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Сохранение базовых изображений


сохраняем 10к изображений из MJSynth локально, чтобы дальше работать с фиксированным набором данных и получать одинаковые train/validation выборки при повторном запуске notebook

In [6]:
MJSYNTH_DIR = Path("data/mjsynth")
MJSYNTH_DIR.mkdir(exist_ok=True)

N = 10000

mjsynth = mjsynth.shuffle(seed=42, buffer_size=10000)

for i, x in enumerate(mjsynth):
    if i == N:
        break

    img = x["image"]
    img.save(MJSYNTH_DIR / f"{i}.png")

### Train / Validation split


делим базовые изображения на трейн и валидацию до создания вариантов 0 и 180 градусов, чтобы одно и то же исходное изображение не попало одновременно в обе выборки и чтобы на валидации модель проверять на ранее не виденных изображениях

In [7]:
from sklearn.model_selection import train_test_split

mjsynth_files = sorted(MJSYNTH_DIR.glob("*.png"))

mjsynth_train_files, mjsynth_val_files = train_test_split(
    mjsynth_files,
    test_size=0.2,
    random_state=SEED
)

print("Всего:", len(mjsynth_files))
print("Train:", len(mjsynth_train_files))
print("Val:", len(mjsynth_val_files))

Всего: 10000
Train: 8000
Val: 2000


In [8]:
mjsynth_train_samples = create_orientation_samples(mjsynth_train_files)
mjsynth_val_samples = create_orientation_samples(mjsynth_val_files)

print("train samples:", len(mjsynth_train_samples))
print("val samples:", len(mjsynth_val_samples))

train samples: 16000
val samples: 4000


## русско-английские данные

### загрузка данных

In [9]:
from datasets import load_dataset

bilingual = load_dataset(
    "tehnik-tehnolog/bilingual-ocr-ru-en-synthetic",
    split="train"
)

In [10]:
cyrillic = bilingual.filter(lambda x: x["label"] == "cyrillic")
latin = bilingual.filter(lambda x: x["label"] == "latin")

### сохранение базовых изображений

In [11]:
def save_dataset_sample(dataset, save_dir, prefix, n):
    sample = dataset.shuffle(seed=SEED).select(range(n))

    for i, x in enumerate(sample):
        x["image"].save(save_dir / f"{prefix}_{i}.png")

BILINGUAL_DIR = Path("data/bilingual")
BILINGUAL_DIR.mkdir(exist_ok=True)

save_dataset_sample(cyrillic, BILINGUAL_DIR, "cyrillic", 5000)
save_dataset_sample(latin, BILINGUAL_DIR, "latin", 5000)

### Train / Validation split


In [12]:
bilingual_files = sorted(BILINGUAL_DIR.glob("*.png"))
bilingual_train_files, bilingual_val_files = train_test_split(bilingual_files, test_size=0.2, random_state=SEED)

bilingual_train_samples = create_orientation_samples(bilingual_train_files)
bilingual_val_samples = create_orientation_samples(bilingual_val_files)

print("train:", len(bilingual_train_samples))
print("val:", len(bilingual_val_samples))

train: 16000
val: 4000


# модель

## MobileNetV3-Small

модель берем MobileNetV3-Small тк она компактная и быстрая, что хорошо подходит под требования задачи

меняем последний слой у модели, тк по дефолту там стоит на выход 1000, а нам требуется 1

In [13]:
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

def create_model():
    weights = MobileNet_V3_Small_Weights.DEFAULT
    model = mobilenet_v3_small(weights=weights)

    model.classifier[3] = nn.Linear(1024, 1)

    return model

# эксперименты


## функция обучения и валидации

функция обучает модель и после каждой эпохи проверяет ее на валидации. В качестве основной метрики используем 1 - Brier Score После завершения обучения загружаем лучшие найденные веса

In [14]:
def train_and_validate_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    loss_fn,
    save_path,
    epochs=5
):
    best_score = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = loss_fn(logits, labels)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)
        model.eval()

        total_brier = 0
        total_count = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.float().unsqueeze(1).to(device)

                logits = model(images)
                probs = torch.sigmoid(logits)

                total_brier += ((probs - labels) ** 2).sum().item()
                total_count += labels.numel()

        brier = total_brier / total_count
        score = 1 - brier

        print(f"epoch {epoch + 1}: " f"train loss = {train_loss:.4f}, "f"1 - Brier = {score:.4f}")

        if score > best_score:
            best_score = score
            torch.save(model.state_dict(), save_path)
    
    model.load_state_dict(torch.load(save_path, map_location=device))

    return model, best_score

## эксперимент 1 MJSynth


обучаем MobileNetV3-Small только на MJSynth и оцениваем качество на MJSynth валидации

### Dataset и DataLoader



создаем трейн и валидационный датасеты и DataLoader для батч загрузки данных

In [15]:
mjsynth_train_dataset = OrientationDataset(
    mjsynth_train_samples,
    train_transform
)

mjsynth_val_dataset = OrientationDataset(
    mjsynth_val_samples,
    val_transform
)

mjsynth_train_loader = DataLoader(
    mjsynth_train_dataset,
    batch_size=32,
    shuffle=True
)

mjsynth_val_loader = DataLoader(
    mjsynth_val_dataset,
    batch_size=32,
    shuffle=False
)

print("train:", len(mjsynth_train_dataset))
print("val:", len(mjsynth_val_dataset))

train: 16000
val: 4000


### модель


создаем MobileNetV3-Small для  эксперимента на MJSynth данных

In [16]:
mjsynth_model = create_model()


### loss и optimizer

переносим модель на доступное устройство. для обучения используем бинарную кросс энтропию, тк задача сводится к бинарной классификации, а optimizer будет обновлять веса модели

In [17]:
mjsynth_model = mjsynth_model.to(device)
mjsynth_loss_fn = nn.BCEWithLogitsLoss()

mjsynth_optimizer = torch.optim.Adam(mjsynth_model.parameters(), lr=0.001)

### обучение и валидация

запускаем train_and_validate_model для обучения и валидации. лучшие параметры сохраняем

In [18]:
mjsynth_model, mjsynth_best_score = train_and_validate_model(
    mjsynth_model,
    mjsynth_train_loader,
    mjsynth_val_loader,
    mjsynth_optimizer,
    mjsynth_loss_fn,
    "model_mjsynth_best.pth",
    epochs=5
)

print("best score:", mjsynth_best_score)

epoch 1: train loss = 0.4699, 1 - Brier = 0.9086
epoch 2: train loss = 0.3005, 1 - Brier = 0.9325
epoch 3: train loss = 0.2483, 1 - Brier = 0.9442
epoch 4: train loss = 0.2198, 1 - Brier = 0.9517
epoch 5: train loss = 0.2033, 1 - Brier = 0.9489
best score: 0.9516968590389006


## эксперимент 2 русско-английские данные

### Dataset и DataLoader

создаем трейн и валидационный датасеты и DataLoader для батч загрузки данных

In [19]:
bilingual_train_dataset = OrientationDataset(
    bilingual_train_samples,
    train_transform
)

bilingual_val_dataset = OrientationDataset(
    bilingual_val_samples,
    val_transform
)

bilingual_train_loader = DataLoader(
    bilingual_train_dataset,
    batch_size=32,
    shuffle=True
)

bilingual_val_loader = DataLoader(
    bilingual_val_dataset,
    batch_size=32,
    shuffle=False
)

print("train:", len(bilingual_train_dataset))
print("val:", len(bilingual_val_dataset))

train: 16000
val: 4000


### модель

создаем новую MobileNetV3-Small для  эксперимента на русско-английских данных

In [20]:
bilingual_model = create_model()

### loss и optimizer

переносим модель на доступное устройство. для обучения используем бинарную кросс энтропию, тк задача сводится к бинарной классификации, а optimizer будет обновлять веса модели

In [21]:
bilingual_model = bilingual_model.to(device)

bilingual_loss_fn = nn.BCEWithLogitsLoss()

bilingual_optimizer = torch.optim.Adam(bilingual_model.parameters(), lr=0.001)

### обучение и валидация

запускаем train_and_validate_model для обучения и валидации. лучшие параметры сохраняем

In [22]:
bilingual_model, bilingual_best_score = train_and_validate_model(
    bilingual_model,
    bilingual_train_loader,
    bilingual_val_loader,
    bilingual_optimizer,
    bilingual_loss_fn,
    "model_bilingual_best.pth",
    epochs=5
)

print("best score:", bilingual_best_score)

epoch 1: train loss = 0.3170, 1 - Brier = 0.9283
epoch 2: train loss = 0.2291, 1 - Brier = 0.9361
epoch 3: train loss = 0.1955, 1 - Brier = 0.9323
epoch 4: train loss = 0.1751, 1 - Brier = 0.9558
epoch 5: train loss = 0.1495, 1 - Brier = 0.9630
best score: 0.9629993400002131


## кросс-проверка моделей

проверяем каждую модель на валидационных данных другого датасета, чтобы оценить переносимость между разными типами данных

In [23]:
def evaluate_model(model, data_loader):
    model.eval()

    total_brier = 0
    total_count = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            logits = model(images)
            probs = torch.sigmoid(logits)

            total_brier += ((probs - labels) ** 2).sum().item()
            total_count += labels.numel()

    brier = total_brier / total_count

    return 1 - brier

In [24]:
def evaluate_model_tta(model, data_loader):
    model.eval()

    total_brier = 0
    total_count = 0 

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            logits_original = model(images)
            p_original = torch.sigmoid(logits_original)

            rotated_images = torch.flip(images, dims=[2, 3])

            logits_rotated = model(rotated_images)
            p_rotated = torch.sigmoid(logits_rotated)

            probs = (p_original + (1 - p_rotated)) / 2

            total_brier += ((probs - labels) ** 2).sum().item()
            total_count += labels.numel()

    brier = total_brier / total_count

    return 1 - brier

In [25]:
mjsynth_on_bilingual = evaluate_model(
    mjsynth_model,
    bilingual_val_loader
)

bilingual_on_mjsynth = evaluate_model(
    bilingual_model,
    mjsynth_val_loader
)

print("MJSynth model → bilingual val:", mjsynth_on_bilingual)
print("Bilingual model → MJSynth val:", bilingual_on_mjsynth)

MJSynth model → bilingual val: 0.826630432844162
Bilingual model → MJSynth val: 0.8390956615805626


обе модели хорошо показывают себя на своих валидационных данных, но сильно просаживаются на кросс данных. Модель на русско-английских данных лучше переносится на MJSynth,  что показывает bilingual модель лучше переносится на другие данные 

## эксперимент 3 смесь MJSynth и русско-английских данных

### Dataset и DataLoader

объединяем обучающие и валидационные данные двух датасетов

In [26]:
mixed_train_samples = mjsynth_train_samples + bilingual_train_samples
mixed_val_samples = mjsynth_val_samples + bilingual_val_samples

mixed_train_dataset = OrientationDataset(
    mixed_train_samples,
    train_transform
)

mixed_val_dataset = OrientationDataset(
    mixed_val_samples,
    val_transform
)

mixed_train_loader = DataLoader(
    mixed_train_dataset,
    batch_size=32,
    shuffle=True
)

mixed_val_loader = DataLoader(
    mixed_val_dataset,
    batch_size=32,
    shuffle=False
)

print("train:", len(mixed_train_dataset))
print("val:", len(mixed_val_dataset))

train: 32000
val: 8000


### модель

создаем MobileNetV3-Small для эксперимента на смеси двух датасетов

In [27]:
mixed_model = create_model()

### loss и optimizer

переносим модель на доступное устройство. для обучения используем бинарную кросс энтропию, тк задача сводится к бинарной классификации, а optimizer будет обновлять веса модели

In [28]:
mixed_model = mixed_model.to(device)
mixed_loss_fn = nn.BCEWithLogitsLoss()
mixed_optimizer = torch.optim.Adam(mixed_model.parameters(), lr=0.001)

### обучение и валидация

запускаем train_and_validate_model для обучения и валидации. лучшие параметры сохраняем

In [29]:
mixed_model, mixed_best_score = train_and_validate_model(
    mixed_model,
    mixed_train_loader,
    mixed_val_loader,
    mixed_optimizer,
    mixed_loss_fn,
    "model_mix_best.pth",
    epochs=8
)

print("best score:", mixed_best_score)

epoch 1: train loss = 0.3864, 1 - Brier = 0.9222
epoch 2: train loss = 0.2673, 1 - Brier = 0.9338
epoch 3: train loss = 0.2167, 1 - Brier = 0.9525
epoch 4: train loss = 0.1868, 1 - Brier = 0.9390
epoch 5: train loss = 0.1706, 1 - Brier = 0.9596
epoch 6: train loss = 0.1560, 1 - Brier = 0.9549
epoch 7: train loss = 0.1499, 1 - Brier = 0.9663
epoch 8: train loss = 0.1402, 1 - Brier = 0.9670
best score: 0.9669890041164435


### проверка на отдельных validation-выборках

отдельно проверяем mixed-модель на валидационных данных MJSynth и bilingual, чтобы оценить качество по отдельности

In [30]:
mixed_on_mjsynth = evaluate_model(
    mixed_model,
    mjsynth_val_loader
)

mixed_on_bilingual = evaluate_model(
    mixed_model,
    bilingual_val_loader
)

print("Mixed model → MJSynth val:", mixed_on_mjsynth)
print("Mixed model → bilingual val:", mixed_on_bilingual)

Mixed model → MJSynth val: 0.9583775703747451
Mixed model → bilingual val: 0.975600437858142


модель на смеси данных показывает стабильный результат на обоих валидационных датасетах: 0.9583775703747451 на MJSynth и 0.975600437858142 на русско-английских данных. В отличие от моделей, обученных на одном датасете, сильной просадки между двумя доменами нет

### проверка с TTA

используем TTA: делаем предсказание для исходного изображения и для этого же изображения, повернутого на 180 градусов. После поворота класс должен поменяться, поэтому объединяем две вероятности. Это делает итоговое предсказание более устойчивым

In [31]:
mixed_tta_mjsynth = evaluate_model_tta(
    mixed_model,
    mjsynth_val_loader
)

mixed_tta_bilingual = evaluate_model_tta(
    mixed_model,
    bilingual_val_loader
)

print("TTA → MJSynth val:", mixed_tta_mjsynth)
print("TTA → bilingual val:", mixed_tta_bilingual)

TTA → MJSynth val: 0.965938110938092
TTA → bilingual val: 0.9812427685102448


# сравнение результатов

In [32]:
print("MJSynth model:")
print("MJSynth val:", mjsynth_best_score)
print("bilingual val:", mjsynth_on_bilingual)

print()

print("bilingual model:")
print("MJSynth val:", bilingual_on_mjsynth)
print("bilingual val:", bilingual_best_score)

print()

print("mixed model:")
print("MJSynth val:", mixed_on_mjsynth)
print("bilingual val:", mixed_on_bilingual)

MJSynth model:
MJSynth val: 0.9516968590389006
bilingual val: 0.826630432844162

bilingual model:
MJSynth val: 0.8390956615805626
bilingual val: 0.9629993400002131

mixed model:
MJSynth val: 0.9583775703747451
bilingual val: 0.975600437858142


mixed модель показывает наиболее стабильное качество на обоих валидационных датасетах, поэтому используем ее как основную. Для финального решения также применяем усиленные аугментации при обучении и TTA при предсказании

# предсказания на тестовых данных


## подготовка тестовых данных

In [33]:
DATA_DIR = Path("data")
TEST_DIR = DATA_DIR / "test" / "test" / "images"
SAMPLE_SUB_PATH = DATA_DIR / "test" / "sample_submission.csv"

sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

In [34]:
test_files = [TEST_DIR / f"{image_id}.png" for image_id in sample_sub["image_id"]]


class TestDataset(Dataset):

    def __init__(self, files):
        self.files = files

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        path = self.files[i]
        img = Image.open(path).convert("RGB")
        img = resize_with_padding(img)
        img = val_transform(img)

        return img


test_dataset = TestDataset(test_files)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("test:", len(test_dataset))

test: 20000


## получение предсказаний

In [35]:
mixed_model.eval()

predictions_tta = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)

        logits_original = mixed_model(images)
        p_original = torch.sigmoid(logits_original)

        rotated_images = torch.flip(images, dims=[2, 3])

        logits_rotated = mixed_model(rotated_images)
        p_rotated = torch.sigmoid(logits_rotated)

        probs = (p_original + (1 - p_rotated)) / 2

        predictions_tta.extend(probs.squeeze(1).cpu().numpy())

print("predictions:", len(predictions_tta))

predictions: 20000


# создание submission

формируем итоговый submission.csv в порядке изображений из sample_submission.csv. В колонку p_180 записываем вероятности, полученные с помощью TTA

In [36]:
submission = sample_sub.copy()
submission["p_180"] = predictions_tta

submission.to_csv(
    "submission.csv",
    index=False
)

In [37]:
print("shape:", submission.shape)
print("NaN:", submission["p_180"].isna().sum())
print("min:", submission["p_180"].min())
print("max:", submission["p_180"].max())

shape: (20000, 2)
NaN: 0
min: 7.432386e-14
max: 1.0


In [38]:
torch.save(
    mixed_model.state_dict(),
    "model_final.pth"
)

# итог

использовали два внешних OCR-датасета: MJSynth и bilingual ru en. Для каждого создали обьекты с поворотом 0 и 180 градусов.  
В качестве модели использовали pretrained MobileNetV3-Small из-за ее компактности и скорости. Для уменьшения различий между синтетическими и реальными OCR-кропами добавили аугментации: блюр, изменение яркости и контраста, небольшие геометрические искажения  
Модели сначала проверяли отдельно и кросс валидировали между двумя датасетами, а также после обучили итоговую модель на комбинации первых двух(MJSynth и bilingual). При инференсе использовали TTA с поворотом изображения на 180°.  

результат на Stepik: 1 - Brier Score = 0.86516010.